# Day C3 — Fourier 변환과 power spectrum: 해상도, 테이퍼, Welch, 다중 테이퍼

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/C-neural-signals/day-C3.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/C-neural-signals/day-C3.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 느린 Fourier 변환으로 진폭과 위상 읽기

In [ ]:
import numpy as np

fs = 100
t = np.arange(0, 2, 1 / fs)              # 2초, N = 200
N = len(t)
x = 1.5 + 2.0 * np.sin(2 * np.pi * 5 * t) + 0.7 * np.cos(2 * np.pi * 12 * t + 1.0)

# 느린 Fourier 변환: 주파수마다 복소 사인파와의 내적
dft = np.zeros(N, dtype=complex)
for k in range(N):
    csw = np.exp(-2j * np.pi * k * np.arange(N) / N)
    dft[k] = x @ csw
print("손으로 만든 DFT == np.fft.fft ?", np.allclose(dft, np.fft.fft(x)))

X = np.fft.rfft(x)                       # 0 Hz ~ Nyquist 만
hz = np.fft.rfftfreq(N, 1 / fs)
amp = np.abs(X) / N
amp[1:-1] *= 2                           # 양(+)과 음(-) 주파수에 나뉜 몫을 합친다. 0 Hz와 Nyquist는 제외
phase = np.angle(X)

print("주파수 간격(해상도) = fs/N = %.2f Hz, 칸 수 = %d" % (fs / N, len(hz)))
for f in [0, 5, 12]:
    i = np.argmin(np.abs(hz - f))
    print("%4.1f Hz: 진폭 %.3f, 위상 %+.2f rad" % (hz[i], amp[i], phase[i]))

### 예시 2 — zero-padding은 두 봉우리를 갈라 주지 않는다

In [ ]:
import numpy as np
from scipy.signal import find_peaks

fs = 250
def two_peaks(dur, pad_to=None):
    t = np.arange(0, dur, 1 / fs)
    x = np.sin(2 * np.pi * 10 * t) + np.sin(2 * np.pi * 10.5 * t)   # 0.5 Hz 떨어진 두 리듬
    nfft = pad_to or len(x)
    amp = np.abs(np.fft.rfft(x, nfft))
    hz = np.fft.rfftfreq(nfft, 1 / fs)
    sel = (hz > 8) & (hz < 13)
    pk, _ = find_peaks(amp[sel], prominence=amp[sel].max() * 0.1)
    return round(fs / len(x), 3), round(fs / nfft, 3), np.round(hz[sel][pk], 2)

for dur, pad in [(1, None), (1, 250 * 8), (4, None)]:
    res, step, pk = two_peaks(dur, pad)
    print("기록 %d초, FFT 점 %5s: 해상도 %.3f Hz, 격자 간격 %.3f Hz, 찾은 봉우리 %s" % (dur, pad or "그대로", res, step, pk))

### 예시 3 — 1/f 배경 위의 alpha: periodogram, Welch, 다중 테이퍼

In [ ]:
import numpy as np
from scipy import signal
from scipy.integrate import simpson

rng = np.random.default_rng(3)
fs = 250
n = fs * 60                                     # 60초 '휴지기' 기록
# 1/f 모양 배경(분홍 잡음): 흰 잡음의 스펙트럼을 1/sqrt(f)로 깎아 만든다
white = np.fft.rfft(rng.normal(size=n))
f_ax = np.fft.rfftfreq(n, 1 / fs)
f_ax[0] = f_ax[1]
pink = np.fft.irfft(white / np.sqrt(f_ax), n)
pink /= pink.std()
t = np.arange(n) / fs
alpha = 0.6 * np.sin(2 * np.pi * 10 * t + rng.uniform(0, 2 * np.pi))
x = pink + alpha

# (1) 한 번에 통째로: periodogram (Hann 테이퍼)
f1, p1 = signal.periodogram(x, fs, window="hann")
# (2) Welch: 2초 창, 50% 겹침, 창마다 Hann 테이퍼 후 평균
f2, p2 = signal.welch(x, fs, nperseg=2 * fs, noverlap=fs)
# (3) 다중 테이퍼: 2초 조각 하나에 서로 직교하는 DPSS 테이퍼 5개(NW=3)를 씌워 평균
seg = x[:2 * fs]
tapers = signal.windows.dpss(2 * fs, NW=3, Kmax=5)
mt = np.mean([np.abs(np.fft.rfft(seg * w)) ** 2 for w in tapers], axis=0) / fs
f3 = np.fft.rfftfreq(2 * fs, 1 / fs)
f4, p4 = signal.periodogram(seg, fs, window="hann")

def roughness(f, p, lo=20, hi=40):
    """20–40 Hz 구간에서 이웃 칸끼리 log 파워가 얼마나 들쭉날쭉한가 (작을수록 매끈)"""
    s = (f >= lo) & (f <= hi)
    return np.std(np.diff(np.log10(p[s])))

print("주파수 간격: periodogram(60초) %.3f Hz | Welch(2초 창) %.3f Hz" % (f1[1], f2[1]))
print("들쭉날쭉함: periodogram %.2f | Welch %.2f" % (roughness(f1, p1), roughness(f2, p2)))
print("2초 조각 하나: Hann 한 장 %.2f | 다중 테이퍼 5장 %.2f" % (roughness(f4, p4), roughness(f3, mt)))

# 대역 파워: alpha(8–12 Hz)를 Welch 스펙트럼에서 적분
def band_power(f, p, lo, hi):
    s = (f >= lo) & (f <= hi)
    return simpson(p[s], x=f[s])
total = band_power(f2, p2, 1, 40)
a = band_power(f2, p2, 8, 12)
print("alpha 절대 파워 %.3f, 1–40 Hz 대비 상대 파워 %.1f%%" % (a, 100 * a / total))
print("Welch 최고점(5–20 Hz): %.1f Hz" % f2[(f2 > 5) & (f2 < 20)][np.argmax(p2[(f2 > 5) & (f2 < 20)])])